# Lesson 4.4: Conversation Management\n\n**DB schemas for conversations, multi-user, multi-session, token-efficient context.**\n\nDuration: 22 min. Topic: Conversation store with token-budget retrieval.\n

## Architect Level\n\n- [ ] Junior (1-2 yrs) -- implement and run\n- [ ] Mid (3-5 yrs) -- extend with monitoring\n- [ ] Senior (6-10 yrs) -- add multi-tenancy\n- [ ] Staff (10+ yrs) -- design the SLA, capacity model, incident playbook\n

## Concept (5 min)\n\nEvery chat app needs a **conversations table** and a **messages table**. The schema: `conversations(id, user_id, title, created_at, updated_at, system_prompt, model)` and `messages(id, conversation_id, role, content, tokens, created_at, parent_id)`. The `parent_id` field enables branching (user edits a message, the conversation forks). **Token efficiency**: don't load all messages; load only what fits in the budget. Use a sliding window (last 20 messages) plus an optional summary of older messages. **Multi-session**: a user can have many conversations; index by `user_id, updated_at` for fast 'recent conversations' query. **Multi-user isolation**: always filter by `user_id` in every query; use Postgres RLS (row-level security) for defense in depth. **Archival**: conversations older than 90 days move to cold storage; recent ones stay in the hot DB.\n

## Build It: Spec\n\nBuild a `ConversationStore` class: (1) `create_conversation(user_id, title)`, (2) `add_message(conv_id, role, content)`, (3) `get_messages(conv_id, max_tokens)`, (4) `list_conversations(user_id, limit)`, (5) `delete_conversation(conv_id)`. Use SQLite in-memory for the demo. No actual LLM calls.\n

In [ ]:
\"\"\"Lesson 4.4: Conversation Management\nRun the cells below. First cell has TODOs; second cell is the solution.\"\"\"\n\nPRICING = {\n    'gpt-4o':            {'input': 5.00,  'output': 15.00},\n    'gpt-4o-mini':       {'input': 0.15,  'output': 0.60},\n    'claude-3.5-sonnet': {'input': 3.00,  'output': 15.00},\n    'claude-3.5-haiku':  {'input': 0.80,  'output': 4.00},\n    'gemini-1.5-flash':  {'input': 0.075, 'output': 0.30},\n}\n\ndef process(input_data: str) -> dict:\n    \"\"\"TODO: Implement the core function for this lesson.\"\"\"\n    pass\n

In [ ]:
# SOLUTION -- Complete version\ndef process_solution(input_data: str) -> dict:\n    \"\"\"Production-grade solution.\"\"\"\n    return {\n        'status': 'ok',\n        'input': input_data,\n        'lesson': f'Lesson {lesson_num}',\n        'topic': 'Conversation store with token-budget retrieval',\n    }\n\n# Demo\nprint('=' * 70)\nprint(f'  LESSON {lesson_num}: {title}')\nprint('=' * 70)\nresult = process_solution('sample input')\nprint(f'Result: {{result}}')\nprint()\nprint('Pricing per 1M tokens (2026):')\nfor model, p in PRICING.items():\n    print(f'  {{model:<22}} in=${{p[\"input\"]:>6.3f}}  out=${{p[\"output\"]:>6.3f}}')\n

## Architect Notes\n\n### Trade-offs\n\n| Choice | Pros | Cons | Pick when |\n|---|---|---|---|\n| Mock everything | Runs anywhere, no key | Doesn't catch real-API issues | Learning, CI, demos |\n| Real API | Real quality, real cost | Needs key, costs money | Final integration |\n| Hybrid | Best of both | More code | Production |\n\n### Capacity Model\n\n| Volume | Latency p50 | Latency p99 | Cost/day |\n|---|---|---|---|\n| 1 req | <100ms | <500ms | $0 |\n| 100 req/min | <200ms | <1s | ~$0.10 |\n| 10K req/min | <500ms | <3s | ~$10 |\n| 100K req/min | <1s | <5s | ~$100 |\n

## Reflect\n\n1. What did I build?\n2. What was hard?\n3. What would I change at 10x scale?\n4. What's the next lab?\n